# Topological Condemnation: Structural Stability and Kinematic Detection of Epidemiological Waves in Sobolev Spaces

**Author:** Santi García-Cremades (Miguel Hernandez University of Elche)  
**Associated Paper:** *Redefining Epidemiological Waves: Structural Stability and Global Empirical Validation in Sobolev Spaces*

### Overview
This notebook provides the complete, reproducible pipeline for detecting structural epidemic waves from highly noisy institutional data. By shifting the paradigm from absolute infection counts to the geometric duration of structural instability—a metric we term **Topological Condemnation**—this framework offers a domain-agnostic, purely mathematical approach to wave detection.

### Core Mathematical Framework
Raw epidemiological data is fundamentally non-stationary and corrupted by institutional artifacts (e.g., weekend under-reporting). Direct differentiation of such data amplifies high-frequency noise quadratically, making peak detection structurally unstable.

To restore geometric stability, we project the raw incidence series $Y$ into the Sobolev space $H^3$ by solving the following discrete Tikhonov regularization problem:

$$\hat{x} = \arg\min_{x \in \mathbb{R}^n} \|x - Y\|_2^2 + \lambda \|\nabla^3 x\|_2^2$$

Once the continuous state $\hat{x}$ is recovered, epidemic waves are defined topologically through the exact zero-crossings of the kinematic derivatives (velocity $v_t$ and acceleration $a_t$). We strictly anchor the wave candidate persistence to $L = 14$ days to reflect the clinical and transmission cycle of SARS-CoV-2.

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os
import shutil
from IPython.display import display

warnings.filterwarnings('ignore')

# =============================================================================
# 1. DIRECTORY STRUCTURE
# =============================================================================
MASTER_DIR = 'Epidemiological_Results'
OUTPUT_DIR = f'{MASTER_DIR}/DASHBOARDS_H3'

for directory in [MASTER_DIR, OUTPUT_DIR]:
    if not os.path.exists(directory):
        os.makedirs(directory)

# =============================================================================
# 2. HYPERPARAMETERS & SENSITIVITY GRID (N = 20)
# =============================================================================
POPULATION_NORM = 100000        

# Baseline parameters for the master wave detection
BASE_LAMBDA = 5000           
BASE_AMIN = 50           
BASE_L = 14 # FIXED: Epidemiological cycle of 14 days (incubation + resolution)

# Sensitivity Grid (Theta) -> 5 x 4 = 20 configurations for the Stability Index (S)
GRID_LAMBDAS = [1000, 5000, 10000, 50000, 100000]
GRID_AMINS = [25, 50, 100, 250]
DELTA_DAYS = 10 # Tolerance for peak matching across grid configurations

# =============================================================================
# 3. GLOBAL DATA ACQUISITION
# =============================================================================
print("Fetching global COVID-19 time series from JHU CSSE...")
url_cases = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
df_raw = pd.read_csv(url_cases)

print("Fetching demographic mapping data...")
url_pop = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/UID_ISO_FIPS_LookUp_Table.csv"
df_pop = pd.read_csv(url_pop)

# Aggregate provincial data into national macro-series
df_grouped = df_raw.groupby('Country/Region').sum(numeric_only=True)
date_cols = [col for col in df_grouped.columns if '/' in col]
df_ts = df_grouped[date_cols]
dates = pd.to_datetime(date_cols)

# Build a robust population dictionary
pop_dict = df_pop[df_pop['Province_State'].isna()].set_index('Country_Region')['Population'].to_dict()
pop_dict['US'] = 331000000
pop_dict['Korea, South'] = 51269183

Fetching global COVID-19 time series from JHU CSSE...
Fetching demographic mapping data...


In [2]:
# =============================================================================
# 4. MATHEMATICAL ENGINE & STABILITY EVALUATION
# =============================================================================

def apply_h3_regularization(data, lam):
    """
    Projects the raw time series into the Sobolev space H^3
    using sparse matrix operations for O(n) computational efficiency.
    """
    n = len(data)
    e = np.ones(n)
    D2 = sparse.spdiags([e, -2*e, e], [-1, 0, 1], n, n)
    H = sparse.eye(n) + lam * (D2.T @ D2)
    return spsolve(H.tocsr(), data)

def detect_waves_rigorous(incidence_smoothed, a_min, l_dur=BASE_L):
    """
    Identifies structurally stable epidemic waves based strictly on the 
    zero-crossings of velocity (v=0) in the regularized H^3 space.
    """
    velocity = np.gradient(incidence_smoothed)
    
    # Pure kinematic zero-crossings
    v_sign = np.sign(velocity)
    v_sign[v_sign == 0] = 1 # Edge case handler for exact zeros
    
    peaks = np.where((v_sign[:-1] > 0) & (v_sign[1:] < 0))[0]
    valleys = np.where((v_sign[:-1] < 0) & (v_sign[1:] > 0))[0]
    
    waves = []
    for peak in peaks:
        valid_valleys_before = valleys[valleys < peak]
        valid_valleys_after = valleys[valleys > peak]
        
        # Anchor the wave strictly from the preceding valley to the succeeding valley
        start = valid_valleys_before[-1] if len(valid_valleys_before) > 0 else 0
        end = valid_valleys_after[0] if len(valid_valleys_after) > 0 else len(incidence_smoothed) - 1
        
        amplitude = incidence_smoothed[peak] - incidence_smoothed[start]
        duration = end - start
        
        if amplitude >= a_min and duration >= l_dur:
            waves.append({'start': start, 'peak': peak, 'end': end, 'val': incidence_smoothed[peak], 'amp': amplitude})
            
    # Remove overlapping days to ensure isolated, mathematically sound intervals
    for i in range(len(waves) - 1):
        if waves[i]['end'] >= waves[i+1]['start']:
            waves[i]['end'] = waves[i+1]['start'] - 1
            
    return waves

def calculate_stability_and_grid(ia14_raw, baseline_waves):
    """
    Evaluates every detected wave against the Theta sensitivity grid (N=20).
    Returns the Stability Index (S) and a 2D matrix for heatmap generation.
    """
    N_configs = len(GRID_LAMBDAS) * len(GRID_AMINS)
    all_config_peaks = []
    wave_counts_grid = np.zeros((len(GRID_LAMBDAS), len(GRID_AMINS)))
    
    # Sweep through the Cartesian product of Lambda and Amin
    for i, lam in enumerate(GRID_LAMBDAS):
        ia14_h3_test = apply_h3_regularization(ia14_raw, lam)
        for j, Amin in enumerate(GRID_AMINS):
            waves_test = detect_waves_rigorous(ia14_h3_test, Amin, BASE_L)
            all_config_peaks.append([w['peak'] for w in waves_test])
            wave_counts_grid[i, j] = len(waves_test)
                
    stability_indices = []
    for bw in baseline_waves:
        # Check peak proximity across all grid configurations
        hits = sum(1 for config_peaks in all_config_peaks if any(abs(p - bw['peak']) <= DELTA_DAYS for p in config_peaks))
        stability_indices.append(round(hits / N_configs, 3))
        
    return stability_indices, wave_counts_grid

In [ ]:
# =============================================================================
# 5. GLOBAL BATCH PROCESSING & DASHBOARD GENERATION
# =============================================================================
import gc # ¡NUEVO! Recolector de basura para liberar RAM

countries_to_analyze = df_ts.index.tolist()
total_countries = len(countries_to_analyze)

summary_data = []
detailed_waves_data = []

# INTERRUPTOR: Ponlo en False si solo quieres generar los CSVs súper rápido sin consumir disco
GENERATE_PLOTS = True 

print(f"\nInitiating Topo-Kinematic Analysis for {total_countries} nations (20 configs/country)...")

for idx_c, country in enumerate(countries_to_analyze):
    safe_name = country.replace("*", "").replace("/", "_")
    
    pop = pop_dict.get(country, None)
    if pop is None or pop < 100000: # Filter micro-states
        continue
        
    cumulative_cases = df_ts.loc[country].values
    daily_cases = np.diff(cumulative_cases, prepend=0)
    daily_cases[daily_cases < 0] = 0 
    
    total_cases_pandemic = int(np.sum(daily_cases))
    
    # Calculate IA14 (Incidence per 100k)
    ia14_raw = np.convolve(daily_cases, np.ones(14), 'valid') / (pop / POPULATION_NORM)
    ia14_raw = np.pad(ia14_raw, (len(cumulative_cases) - len(ia14_raw), 0), 'constant')
    
    # Base Regularization
    ia14_h3 = apply_h3_regularization(ia14_raw, BASE_LAMBDA)
    baseline_waves = detect_waves_rigorous(ia14_h3, BASE_AMIN, BASE_L)
    total_waves = len(baseline_waves)
    
    if total_waves == 0:
        summary_data.append({
            "Country": country, "Total_Waves": 0, "Major_Waves_Count": 0, 
            "Avg_Stability_Index": 0.0, "Total_Infected_In_Waves": 0, 
            "Total_Infected_Pandemic": total_cases_pandemic, "Pct_Cases_In_Waves": "0.00%",
            "Total_Epidemic_Days": 0
        })
        continue
        
    stability_indices, wave_counts_grid = calculate_stability_and_grid(ia14_raw, baseline_waves)
    avg_S = round(np.mean(stability_indices), 2)
    
    total_infected_waves = 0
    total_epidemic_days = 0
    major_waves_count = 0
    
    # Pre-calculamos los datos para las tablas (independientemente de si dibujamos o no)
    for w_idx, w in enumerate(baseline_waves):
        s, p, e = w['start'], w['peak'], w['end']
        S_i = stability_indices[w_idx]
        is_major = S_i > 0.8
        
        if is_major: major_waves_count += 1
            
        wave_infected = int(np.sum(daily_cases[s:e+1]))
        duration = int((dates[e] - dates[s]).days)
        
        detailed_waves_data.append({
            "Country": country, "Wave_Number": w_idx + 1,
            "Start_Date": dates[s].strftime('%Y-%m-%d'), "Peak_Date": dates[p].strftime('%Y-%m-%d'), "End_Date": dates[e].strftime('%Y-%m-%d'),
            "Duration_Days": duration, "Peak_to_Onset_Amp": round(w['amp'], 2), "Peak_IA14": round(w['val'], 2),
            "Infected_In_Wave": wave_infected, "Stability_Index_S": S_i, "Is_Major_Wave": is_major
        })
        
        total_infected_waves += wave_infected
        total_epidemic_days += duration

    # ---------------- COMBINED DASHBOARD RENDERING ----------------
    if GENERATE_PLOTS:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [2.5, 1]}, constrained_layout=True)
        fig.suptitle(f"Topological Epidemic Analysis: {country}", fontsize=20, fontweight='bold', ha='center')
        
        # --- AXIS 1: The Epidemic Wave Plot ---
        ax1.plot(dates, ia14_raw, color='gray', alpha=0.3, label="Real IA14")
        ax1.plot(dates, ia14_h3, color='black', linewidth=1.5, label="Estimated IA14 ($H^3$ Sobolev)")
        
        for w_idx, w in enumerate(baseline_waves):
            s, p, e, S_i = w['start'], w['peak'], w['end'], stability_indices[w_idx]
            is_major = S_i > 0.8
                
            ax1.fill_between(dates[s:p+1], ia14_h3[s:p+1], 0, color='red', alpha=0.3)
            ax1.fill_between(dates[p:e+1], ia14_h3[p:e+1], 0, color='green', alpha=0.3)
            
            tag_color = "white" if is_major else "#ffe6e6" 
            ax1.annotate(f'Wave {w_idx+1}\n$S = {S_i:.2f}$', 
                        xy=(dates[p], ia14_h3[p]), xytext=(0, 10), textcoords="offset points",
                        ha='center', va='bottom', fontsize=9, fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.3", fc=tag_color, ec="black", lw=1, alpha=0.9))

        ax1.set_title(f"Kinematic Detections ($\lambda={BASE_LAMBDA}$, $L={BASE_L}$ days, $A_{{min}}={BASE_AMIN}$) | Avg S-Index: {avg_S}", fontsize=14, ha='center')
        ax1.set_ylabel("Cumulative Incidence 14-days (per 100k)", fontsize=12)
        ax1.set_ylim(bottom=0)
        ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        ax1.tick_params(axis='x', rotation=45)

        legend_elements = [
            mpatches.Patch(facecolor='red', alpha=0.3, label='Expansion Phase ($v>0$)'),
            mpatches.Patch(facecolor='green', alpha=0.3, label='Mitigation Phase ($v \leq 0$)')
        ]
        handles, labels = ax1.get_legend_handles_labels()
        ax1.legend(handles=handles + legend_elements, loc='upper right')
        
        # --- AXIS 2: The Sensitivity Heatmap ---
        sns.heatmap(wave_counts_grid, annot=True, cmap="YlGnBu", fmt='g', ax=ax2, 
                    xticklabels=GRID_AMINS, yticklabels=GRID_LAMBDAS, cbar_kws={"shrink": .8})
        ax2.set_title(f"Sensitivity Grid: Total Waves Detected (Fixed $L = {BASE_L}$ days)", fontsize=14, ha='center')
        ax2.set_xlabel("Minimum Amplitude ($A_{min}$)", fontsize=12)
        ax2.set_ylabel("Regularization ($\lambda$)", fontsize=12)
                
        # ¡EL TRUCO DE LA MEMORIA! Bajamos DPI a 150 y limpiamos RAM a lo bestia
        plt.savefig(f"{OUTPUT_DIR}/{safe_name}_Dashboard.png", dpi=150, bbox_inches='tight')
        fig.clf() # Limpiamos la figura internamente
        plt.close(fig) # Cerramos la figura explícitamente
        gc.collect() # Forzamos al recolector de basura de Python a vaciar la RAM
    
    pct_in_waves = (total_infected_waves / total_cases_pandemic) * 100 if total_cases_pandemic > 0 else 0
    
    summary_data.append({
        "Country": country, "Total_Waves": total_waves, "Major_Waves_Count": major_waves_count,
        "Avg_Stability_Index": avg_S, "Total_Infected_In_Waves": total_infected_waves,
        "Total_Infected_Pandemic": total_cases_pandemic, "Pct_Cases_In_Waves": f"{pct_in_waves:.2f}%",
        "Total_Epidemic_Days": total_epidemic_days
    })


Initiating Topo-Kinematic Analysis for 201 nations (20 configs/country)...


In [ ]:
# =============================================================================
# 6. EXPORT CSVs & ZIP ARCHIVE
# =============================================================================
print("\nGenerating comprehensive CSV datasets...")

df_summary = pd.DataFrame(summary_data).sort_values(by="Total_Epidemic_Days", ascending=False)
df_summary.to_csv(f"{MASTER_DIR}/Country_Summary_H3.csv", index=False)

df_detailed = pd.DataFrame(detailed_waves_data)
df_detailed.to_csv(f"{MASTER_DIR}/Detailed_Waves_H3.csv", index=False)

print(f"Compressing the '{MASTER_DIR}' folder into a .zip file...")
# This generates 'Epidemiological_Archaeology_Results.zip' with all dashboards and CSVs
shutil.make_archive('Epidemiological_Archaeology_Results', 'zip', MASTER_DIR)

print("\n" + "="*50)
print(" GLOBAL PROCESSING & COMPRESSION COMPLETE!")
print("="*50)
print("-> You can now download 'Epidemiological_Archaeology_Results.zip'.")
display(df_summary.head(10))